In [ ]:
# Import required libraries, load the Telco Customer Churn dataset, and preprocess target/numeric variables
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dataset URL
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Convert TotalCharges to numeric, handle missing values, and map Churn to binary values (0 and 1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_bin'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [ ]:
# Select numeric features and one-hot encode categorical features, then combine them into feature matrix X
num = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']]
cat = pd.get_dummies(df[['Contract', 'PaymentMethod']], drop_first=True)

# Combine datasets and define target variable y
X = pd.concat([num, cat], axis=1)
y = df['Churn_bin']

# Verify the final structure and feature list
print('X shape:', X.shape)
print('Features:', list(X.columns))

In [ ]:
# Calculate Mutual Information scores to find out which features are most relevant for predicting customer churn
from sklearn.feature_selection import mutual_info_classif

# Compute scores and sort them in descending order
mi = mutual_info_classif(X, y, random_state=42)
mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)
mi_scores.round(4)

In [ ]:
# Create a horizontal bar chart to visually compare the Mutual Information scores across all features
plt.figure(figsize=(8, 5))
mi_scores.plot(kind='barh', color='steelblue')
plt.title('Mutual information with Churn')
plt.xlabel('MI score')
plt.tight_layout()
plt.show()

In [ ]:
# Scale the features and use Recursive Feature Elimination (RFE) with Logistic Regression to select the top 3 features
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale data before training
X_scaled = StandardScaler().fit_transform(X)

# Initialize RFE to select exactly 3 features
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=3)
rfe.fit(X_scaled, y)

# Present the results showing which features were kept and their ranking
result = pd.DataFrame({'feature': X.columns, 'kept': rfe.support_, 'rank': rfe.ranking_})
result.sort_values('rank')

In [ ]:
# Split data into train/test sets and compare the performance of Logistic Regression using all features versus only the top 3 features
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train-test split
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Model 1: Using all features
m_all = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print('All features:', round(accuracy_score(y_te, m_all.predict(X_te)), 4))

# Model 2: Using only top 3 features selected by RFE
idx = np.where(rfe.support_)[0]
m_top = LogisticRegression(max_iter=1000).fit(X_tr[:, idx], y_tr)
print('Top 3:', round(accuracy_score(y_te, m_top.predict(X_te[:, idx])), 4))